# 🔧 Feature Engineering — Manufacturing Predictive Maintenance

Prepares features from Silver sensor readings and production batches for
predictive maintenance modeling.

**Reads from:** `silver_sensor_readings`, `silver_production_batches`, `silver_equipment_catalog`

**Writes to:** `gold_ml_features`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, stddev, count,
    sum as spark_sum, max as spark_max, min as spark_min,
    datediff, to_date, expr, lag, abs as spark_abs,
    percentile_approx, window
)
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
# Load Silver tables
sensor_df = spark.read.table('silver_sensor_readings')
batch_df = spark.read.table('silver_production_batches')
equip_df = spark.read.table('silver_equipment_catalog')

print(f'Sensor readings: {sensor_df.count():,} rows')
print(f'Production batches: {batch_df.count():,} rows')
print(f'Equipment catalog: {equip_df.count():,} rows')

In [ ]:
# === Sensor features: aggregate per machine per day ===
sensor_daily = (
    sensor_df
    .groupBy('machine_id', 'reading_date')
    .agg(
        avg('temperature').alias('avg_temp'),
        stddev('temperature').alias('std_temp'),
        spark_max('temperature').alias('max_temp'),
        spark_min('temperature').alias('min_temp'),
        avg('pressure').alias('avg_pressure'),
        stddev('pressure').alias('std_pressure'),
        spark_max('pressure').alias('max_pressure'),
        avg('vibration').alias('avg_vibration'),
        stddev('vibration').alias('std_vibration'),
        spark_max('vibration').alias('max_vibration'),
        avg('humidity').alias('avg_humidity'),
        count('*').alias('reading_count'),
        spark_sum(when(col('temp_anomaly'), 1).otherwise(0)).alias('temp_anomaly_count'),
        spark_sum(when(col('vibration_anomaly'), 1).otherwise(0)).alias('vib_anomaly_count'),
    )
    .withColumn('temp_range', col('max_temp') - col('min_temp'))
    .withColumn('anomaly_ratio', (col('temp_anomaly_count') + col('vib_anomaly_count')) / col('reading_count'))
)

print(f'Sensor daily features: {sensor_daily.count():,} rows')

In [ ]:
# === Batch features: aggregate per machine per day ===
# Create a target: high_downtime (>60 min in a day = potential maintenance needed)
batch_daily = (
    batch_df
    .withColumn('production_date_col', to_date('production_date'))
    .groupBy('machine_id', 'production_date')
    .agg(
        spark_sum('units_produced').alias('total_units'),
        spark_sum('defect_count').alias('total_defects'),
        spark_sum('downtime_minutes').alias('total_downtime'),
        count('*').alias('batch_count'),
        avg('yield_pct').alias('avg_yield'),
        avg('defect_rate').alias('avg_defect_rate'),
    )
    .withColumn('needs_maintenance',
        when(col('total_downtime') > 60, lit(1)).otherwise(lit(0))
    )
)

maint_rate = batch_daily.filter(col('needs_maintenance') == 1).count() / batch_daily.count() * 100
print(f'Batch daily features: {batch_daily.count():,} rows')
print(f'Maintenance needed rate: {maint_rate:.1f}%')

In [ ]:
# === Join sensor + batch features ===
ml_features = (
    sensor_daily
    .join(batch_daily, ['machine_id', 'reading_date'], 'inner')
    .join(
        equip_df.select('machine_id', 'machine_type', 'production_line', 'install_date'),
        'machine_id', 'left'
    )
    .withColumn('equipment_age_days',
        datediff(to_date('reading_date'), to_date('install_date'))
    )
    .drop('install_date')
    .na.fill(0)
    .withColumn('feature_timestamp', current_timestamp())
)

# Write feature table
ml_features.write.mode('overwrite').format('delta').saveAsTable('gold_ml_features')

print(f'\nGold ML features written: {ml_features.count():,} rows')
print(f'Columns: {len(ml_features.columns)}')
print(f'Target distribution (needs_maintenance):')
ml_features.groupBy('needs_maintenance').count().show()

In [ ]:
spark.sql('OPTIMIZE gold_ml_features')
print('Feature table optimized')